In [ ]:
"""
PASO 3 - Minority Game con abstencion: efecto del numero de agentes N
=====================================================================
Tesis: Complejidad e informacion en un minority game con opcion de abstencion
Base: R. Mansilla, Phys. Rev. E 62, 4553 (2000), Ec. (6)

Alfabeto: 0 = vender, 1 = comprar, 2 = abstenerse

Que hace este script:
  1. Para cada valor de N corre 100 partidas (las mismas 100 semillas
     para todos los N), con agentes de s = 2 estrategias.
  2. A cada cadena de ganadores le aplica M(d), Ec. (6) del PRE 62,
     para d = 1, ..., 100.
  3. Promedia M(d) sobre las 100 cadenas y calcula el intervalo de
     confianza del 95%.
  4. Junta los resultados en dos grupos, y cada grupo en una misma grafica:
        Grupo 1: N = 100, 200, 300, 400
        Grupo 2: N = 1000, 2000, 3000, 4000
     Por grupo salen dos graficas: los promedios, y los promedios con
     sus bandas de confianza.
  5. Guarda CSV por cada N y un CSV de resumen con todos los N.

Parametros fijos: T = 6000, s = 2, m = 3, K = 3, sin burn-in, 100 semillas.
Las funciones de simulacion, de ganador y de M(d) son las mismas del
Segundo paso, sin cambios.

PUNTAJE = True  -> agentes con 2 estrategias CON puntaje (codigo base)
PUNTAJE = False -> agentes con 2 estrategias SIN puntaje
"""

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

try:
    from scipy.stats import t as t_student
    T_95 = None          # se calcula con scipy mas abajo
except ImportError:
    t_student = None
    T_95 = 1.984         # t de Student, 95%, 99 grados de libertad


# =====================================================================
# PARAMETROS
# =====================================================================
SEMILLA_MAESTRA = 1907443528   # genera las 100 semillas de forma reproducible
N_CADENAS = 100                # numero de cadenas (semillas distintas)

M       = 3      # memoria (brain size)
S       = 2      # estrategias por agente
K       = 3      # tamano del alfabeto
T       = 6000   # ciclos registrados, desde el primero (sin burn-in)

PUNTAJE = True   # True = con puntaje, False = sin puntaje

GRUPOS = [
    ("grupo1", [100, 200, 300, 400]),
    ("grupo2", [1000, 2000, 3000, 4000]),
]

D_MAX   = 100    # M(d) se calcula para d = 1, ..., D_MAX
NIVEL   = 0.95   # nivel de confianza del intervalo

ETIQUETA = "con_puntaje" if PUNTAJE else "sin_puntaje"
DESCRIPCION = ("2 estrategias, con puntaje" if PUNTAJE
               else "2 estrategias, sin puntaje")


# =====================================================================
# REGLA DE GANADOR  (identica a la del codigo base)
# =====================================================================
def regla_ganador(conteos, rng):
    """Gana el argmin sobre las K acciones, con desempate aleatorio."""
    empatados = np.flatnonzero(conteos == conteos.min())
    return int(rng.choice(empatados))


# =====================================================================
# s ESTRATEGIAS POR AGENTE, SIN PUNTAJE  (igual que en el Segundo paso)
# =====================================================================
def simular_sin_recompensa(N, m, s, K, T, semilla):
    """Agentes con s estrategias que no las califican: los puntajes
    valen 0 siempre, asi que en cada ciclo cada agente toma al azar una
    de sus estrategias y hace lo que dice para la historia actual.
    """
    rng = np.random.default_rng(semilla)
    N_HIST = K ** m

    # estrategias[i, j, h] = accion que la estrategia j del agente i
    #                        recomienda ante la historia con indice h
    estrategias = rng.integers(0, K, size=(N, s, N_HIST))

    # puntajes[i, j]: se quedan en 0 porque no hay recompensa
    puntajes = np.zeros((N, s))

    # memoria: los m datos que el mercado tiene "al nacer"
    historia = rng.integers(0, K, size=m)

    pesos = K ** np.arange(m - 1, -1, -1)   # convierte historia -> indice
    indices = np.arange(N)

    serie_ganadores = np.zeros(T, dtype=int)

    for t in range(T):

        # 1. que situacion estan viendo los agentes
        idx = int(historia @ pesos)

        # 2. cada agente toma su estrategia con mas puntos.
        #    Como todos los puntajes valen 0, el ruido decide al azar.
        mejor = np.argmax(puntajes + rng.random((N, s)) * 0.5, axis=1)

        # 3. consulta esa estrategia en la historia actual y actua
        acciones = estrategias[indices, mejor, idx]

        # 4. se cuenta cuantos eligieron cada opcion
        conteos = np.bincount(acciones, minlength=K)

        # 5. gana la minoria
        ganador = regla_ganador(conteos, rng)

        # 6. (SIN RECOMPENSA: en el codigo base aqui se sumaban puntos)

        # 7. registrar desde el primer ciclo
        serie_ganadores[t] = ganador

        # 8. la memoria se desplaza: entra el ganador, sale el mas viejo
        historia = np.roll(historia, -1)
        historia[-1] = ganador

    return serie_ganadores


# =====================================================================
# s ESTRATEGIAS POR AGENTE, CON PUNTAJE  (igual que en el Segundo paso)
# =====================================================================
def simular_con_puntaje(N, m, s, K, T, semilla):
    """Reglas del codigo base, sin burn-in. Cada agente usa la estrategia
    con mas puntos y, al final del ciclo, todas las estrategias que
    habrian acertado al ganador reciben un punto.

    Devuelve la serie de ganadores y cuantas veces acerto cada agente.
    """
    rng = np.random.default_rng(semilla)
    N_HIST = K ** m

    # estrategias[i, j, h] = accion que la estrategia j del agente i
    #                        recomienda ante la historia con indice h
    estrategias = rng.integers(0, K, size=(N, s, N_HIST))

    # puntajes[i, j] = puntos virtuales (casos de exito) de esa estrategia
    puntajes = np.zeros((N, s))

    # memoria: los m datos que el mercado tiene "al nacer"
    historia = rng.integers(0, K, size=m)

    pesos = K ** np.arange(m - 1, -1, -1)   # convierte historia -> indice
    indices = np.arange(N)

    serie_ganadores = np.zeros(T, dtype=int)
    exitos_agente = np.zeros(N, dtype=int)

    for t in range(T):

        # 1. que situacion estan viendo los agentes
        idx = int(historia @ pesos)

        # 2. cada agente toma su estrategia con mas puntos.
        #    El ruido < 1 rompe empates al azar sin alterar el orden
        #    entre puntajes enteros distintos.
        mejor = np.argmax(puntajes + rng.random((N, s)) * 0.5, axis=1)

        # 3. consulta esa estrategia en la historia actual y actua
        acciones = estrategias[indices, mejor, idx]

        # 4. se cuenta cuantos eligieron cada opcion
        conteos = np.bincount(acciones, minlength=K)

        # 5. gana la minoria
        ganador = regla_ganador(conteos, rng)

        # 6. se premian TODAS las estrategias que hubieran acertado,
        #    se hayan jugado o no (regla del codigo base / PRE 62).
        puntajes += (estrategias[:, :, idx] == ganador)

        # 7. registrar desde el primer ciclo
        serie_ganadores[t] = ganador
        exitos_agente += (acciones == ganador)

        # 8. la memoria se desplaza: entra el ganador, sale el mas viejo
        historia = np.roll(historia, -1)
        historia[-1] = ganador

    return serie_ganadores, exitos_agente


# =====================================================================
# FUNCION DE INFORMACION MUTUA  -  Ec. (6) del PRE 62
# =====================================================================
def informacion_mutua(serie, d, K):
    """M(d) = sum_{a,b} P_ab(d) * log2[ P_ab(d) / (P_a * P_b) ]

    P_ab(d): probabilidad de encontrar el simbolo a y, d lugares despues,
             el simbolo b. Se estima contando todos los pares
             (serie[i], serie[i+d]) de la cadena.
    P_a:     densidad del simbolo a en la cadena completa (como dice
             Mansilla: "the density of the symbol alpha").

    La suma corre sobre los K*K = 9 pares (a, b). Los pares que nunca
    aparecen (P_ab = 0) no aportan, porque 0 * log(0) = 0.
    """
    x = serie[:-d]          # simbolo alpha
    y = serie[d:]           # simbolo beta, d lugares despues
    n_pares = len(x)

    # probabilidades conjuntas P_ab(d): matriz K x K
    conjunta = np.bincount(x * K + y, minlength=K * K).reshape(K, K) / n_pares

    # densidades P_a de cada simbolo en toda la cadena
    densidad = np.bincount(serie, minlength=K) / len(serie)
    producto = np.outer(densidad, densidad)      # P_a * P_b

    M_d = 0.0
    for a in range(K):
        for b in range(K):
            if conjunta[a, b] > 0:
                M_d += conjunta[a, b] * np.log2(conjunta[a, b] / producto[a, b])
    return M_d


def sesgo_esperado(K, n_pares):
    """Valor que da M(d) en una cadena SIN correlacion, solo por ser finita.

    Es aproximadamente (K-1)^2 / (2 * n * ln 2). Solo depende de T, asi
    que es la misma linea de referencia para todos los N.
    """
    return (K - 1) ** 2 / (2 * n_pares * np.log(2))


def exito_al_azar(N, K, n_muestras=200000, semilla=0):
    """Tasa de exito que tendrian N agentes eligiendo al azar.

    El 0.265 que imprimia el Segundo paso solo vale para N = 50. Aqui se
    calcula para cada N: se reparten N agentes al azar entre las K
    acciones muchas veces y se promedia la fraccion que queda en la
    accion ganadora (la de menos agentes).
    """
    rng = np.random.default_rng(semilla)
    conteos = rng.multinomial(N, [1 / K] * K, size=n_muestras)
    return (conteos.min(axis=1) / N).mean()


# =====================================================================
# CORRER LAS 100 CADENAS PARA UN VALOR DE N
# =====================================================================
def correr_N(N, semillas, distancias):
    tabla = np.zeros((N_CADENAS, D_MAX))     # tabla[c, j] = M(d=j+1)
    frecuencias = np.zeros((N_CADENAS, K))
    tasa_exito = np.zeros(N_CADENAS)

    for c, semilla in enumerate(semillas):
        if PUNTAJE:
            serie, exitos = simular_con_puntaje(N, M, S, K, T, int(semilla))
            tasa_exito[c] = exitos.sum() / (N * T)
        else:
            serie = simular_sin_recompensa(N, M, S, K, T, int(semilla))
            tasa_exito[c] = np.nan
        frecuencias[c] = np.bincount(serie, minlength=K) / T
        for j, d in enumerate(distancias):
            tabla[c, j] = informacion_mutua(serie, d, K)
        if (c + 1) % 10 == 0:
            print(f"  cadenas terminadas: {c + 1}/{N_CADENAS}", flush=True)

    return tabla, frecuencias, tasa_exito


# =====================================================================
# GRAFICAS DE UN GRUPO (todos sus N en la misma grafica)
# =====================================================================
COLORES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]   # uno por cada N
GRIS = "#52514e"
TINTA = "#0b0b0b"


def graficar_grupo(nombre, lista_N, resultados, distancias, referencia):
    plt.rcParams.update({
        "font.size": 11,
        "axes.edgecolor": GRIS,
        "axes.labelcolor": TINTA,
        "xtick.color": GRIS,
        "ytick.color": GRIS,
        "axes.spines.top": False,
        "axes.spines.right": False,
    })
    subtitulo = (f"m={M}, s={S}, K={K}, T={T}, sin burn-in, "
                 f"{N_CADENAS} semillas por cada N")

    # --- Grafica 1: promedios de M(d) para cada N ---
    fig, ax = plt.subplots(figsize=(10, 5.5))
    for N, color in zip(lista_N, COLORES):
        media = resultados[N]["media"]
        ax.plot(distancias, media, color=color, lw=1.8, marker="o", ms=3,
                label=f"N = {N}  (α = {K**M / N:.4f})")
    ax.plot(distancias, referencia, color=GRIS, lw=1.5, ls="--",
            label="Sesgo esperado de una cadena sin correlación")
    ax.set_xlabel("Distancia d")
    ax.set_ylabel("M(d)  [bits]")
    ax.set_title(f"Información mutua promedio — {DESCRIPCION}, "
                 f"N = {', '.join(str(n) for n in lista_N)}\n" + subtitulo,
                 loc="left", color=TINTA)
    ax.set_xlim(0, D_MAX + 1)
    ax.set_ylim(bottom=0)
    ax.grid(axis="y", color="#e5e4df", lw=0.8)
    ax.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
    ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.13),
              ncol=3, fontsize=9)
    fig.tight_layout()
    archivo = f"Paso3_grafica_Md_{ETIQUETA}_{nombre}_promedio.png"
    fig.savefig(archivo, dpi=200)
    plt.close(fig)
    print(f"  Guardado: {archivo}", flush=True)

    # --- Grafica 2: promedios con bandas de confianza ---
    fig, ax = plt.subplots(figsize=(10, 5.5))
    for N, color in zip(lista_N, COLORES):
        r = resultados[N]
        ax.fill_between(distancias, r["ic_inf"], r["ic_sup"], color=color,
                        alpha=0.18, lw=0)
        ax.plot(distancias, r["media"], color=color, lw=1.8,
                label=f"N = {N}  (α = {K**M / N:.4f})")
    ax.plot(distancias, referencia, color=GRIS, lw=1.5, ls="--",
            label="Sesgo esperado de una cadena sin correlación")
    manijas, etiquetas = ax.get_legend_handles_labels()
    manijas.append(Patch(color=GRIS, alpha=0.25, lw=0))
    etiquetas.append(f"Bandas: intervalo de confianza {int(NIVEL*100)}%")
    ax.set_xlabel("Distancia d")
    ax.set_ylabel("M(d)  [bits]")
    ax.set_title(f"Promedio de M(d) con intervalo de confianza del "
                 f"{int(NIVEL*100)}% — {DESCRIPCION}\n"
                 f"N = {', '.join(str(n) for n in lista_N)};  "
                 f"t de Student con {N_CADENAS - 1} grados de libertad",
                 loc="left", color=TINTA)
    ax.set_xlim(0, D_MAX + 1)
    ax.set_ylim(bottom=0)
    ax.grid(axis="y", color="#e5e4df", lw=0.8)
    ax.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
    ax.legend(manijas, etiquetas, frameon=False, loc="upper center",
              bbox_to_anchor=(0.5, -0.13), ncol=3, fontsize=9)
    fig.tight_layout()
    archivo = f"Paso3_grafica_Md_{ETIQUETA}_{nombre}_intervalo_confianza.png"
    fig.savefig(archivo, dpi=200)
    plt.close(fig)
    print(f"  Guardado: {archivo}", flush=True)


# =====================================================================
# PROGRAMA PRINCIPAL
# =====================================================================
if __name__ == "__main__":

    # --- las 100 semillas (las mismas para todos los N) ---
    semillas = np.random.SeedSequence(SEMILLA_MAESTRA).generate_state(N_CADENAS)
    distancias = np.arange(1, D_MAX + 1)

    # --- valor critico de la t de Student ---
    t_crit = (t_student.ppf(0.5 + NIVEL / 2, N_CADENAS - 1)
              if t_student is not None else T_95)

    # --- linea de referencia: igual para todos los N (solo depende de T) ---
    referencia = sesgo_esperado(K, T - distancias)

    resumen = []

    for nombre_grupo, lista_N in GRUPOS:
        resultados = {}

        for N in lista_N:
            alfa = K ** M / N
            print("\n" + "=" * 62)
            print(f"  {DESCRIPCION}  |  {nombre_grupo}")
            print(f"  N={N}  m={M}  s={S}  K={K}  T={T}  burn_in=0"
                  f"  alfa={alfa:.4f}")
            print(f"  cadenas={N_CADENAS}  d = 1..{D_MAX}")
            print("=" * 62, flush=True)

            tabla, frecuencias, tasa_exito = correr_N(N, semillas, distancias)

            # --- promedio sobre las 100 cadenas, para cada d ---
            media = tabla.mean(axis=0)
            desv = tabla.std(axis=0, ddof=1)
            error_est = desv / np.sqrt(N_CADENAS)
            ic_inf = media - t_crit * error_est
            ic_sup = media + t_crit * error_est
            resultados[N] = {"media": media, "ic_inf": ic_inf, "ic_sup": ic_sup}

            tasa_media = tasa_exito.mean() if PUNTAJE else np.nan
            azar = exito_al_azar(N, K)
            d_max = int(distancias[np.argmax(media)])

            # --- resumen en pantalla ---
            print("\n  Frecuencia media de cada simbolo (100 cadenas):")
            for a, nombre in enumerate(["vender", "comprar", "abstenerse"]):
                print(f"    {a} ({nombre:10s}): {frecuencias[:, a].mean():.4f}")
            if PUNTAJE:
                print(f"\n  Tasa de exito media de los agentes: {tasa_media:.4f}"
                      f"  (agentes al azar con N={N}: {azar:.4f})")
            print(f"\n  M(d) promedio: min={media.min():.3e}  "
                  f"max={media.max():.3e} (en d={d_max})  "
                  f"media global={media.mean():.3e} bits")
            print(f"  M(1)={media[0]:.3e}  M(2)={media[1]:.3e}  "
                  f"M(3)={media[2]:.3e}  M(4)={media[3]:.3e}")
            print(f"  Sesgo esperado de una cadena sin correlacion: "
                  f"{referencia.mean():.3e} bits", flush=True)

            # --- guardar resultados de este N ---
            parametros = (f"N={N} m={M} s={S} K={K} T={T} burn_in=0 "
                          f"alfa={alfa:.6f} {ETIQUETA}")
            encabezado = (
                f"PASO 3. M(d) con {DESCRIPCION}, Ec. (6) PRE 62\n"
                f"{parametros} cadenas={N_CADENAS} "
                f"semilla_maestra={SEMILLA_MAESTRA}\n"
                f"intervalo de confianza {int(NIVEL*100)}% con t de Student "
                f"(t={t_crit:.4f})\n"
                f"d,media,desv_estandar,error_estandar,ic_inferior,"
                f"ic_superior,sesgo_esperado"
            )
            np.savetxt(f"Paso3_Md_{ETIQUETA}_N{N}_promedio.csv",
                       np.column_stack([distancias, media, desv, error_est,
                                        ic_inf, ic_sup, referencia]),
                       fmt=["%d"] + ["%.8e"] * 6, delimiter=",",
                       header=encabezado, comments="# ")

            encabezado_todas = (
                f"PASO 3. M(d) de cada cadena. Fila = una semilla, "
                f"columna = d\n{parametros}\n"
                f"semilla," + ",".join(f"d{d}" for d in distancias)
            )
            np.savetxt(f"Paso3_Md_{ETIQUETA}_N{N}_todas_las_cadenas.csv",
                       np.column_stack([semillas.astype(np.int64), tabla]),
                       fmt=["%d"] + ["%.8e"] * D_MAX, delimiter=",",
                       header=encabezado_todas, comments="# ")

            print(f"\n  Guardado: Paso3_Md_{ETIQUETA}_N{N}_promedio.csv")
            print(f"  Guardado: Paso3_Md_{ETIQUETA}_N{N}_todas_las_cadenas.csv",
                  flush=True)

            resumen.append([N, alfa, tasa_media, azar,
                            media[0], media[1], media[2], media[3],
                            media.max(), d_max])

        # --- las graficas del grupo, con todos sus N juntos ---
        print(f"\n  Graficas del {nombre_grupo}:")
        graficar_grupo(nombre_grupo, lista_N, resultados, distancias,
                       referencia)

    # --- resumen de todos los N en un solo archivo ---
    np.savetxt(f"Paso3_resumen_{ETIQUETA}.csv", np.array(resumen),
               fmt=["%d", "%.6f", "%.6f", "%.6f",
                    "%.8e", "%.8e", "%.8e", "%.8e", "%.8e", "%d"],
               delimiter=",",
               header=(f"PASO 3. Resumen, {DESCRIPCION}, m={M} s={S} K={K} "
                       f"T={T} cadenas={N_CADENAS}\n"
                       f"N,alfa,tasa_exito,exito_al_azar,"
                       f"M1,M2,M3,M4,M_max,d_del_maximo"),
               comments="# ")
    print(f"\n  Guardado: Paso3_resumen_{ETIQUETA}.csv")
    print("\n  Listo.")


  2 estrategias, con puntaje  |  grupo1
  N=100  m=3  s=2  K=3  T=6000  burn_in=0  alfa=0.2700
  cadenas=100  d = 1..100
  cadenas terminadas: 10/100
  cadenas terminadas: 20/100
  cadenas terminadas: 30/100
  cadenas terminadas: 40/100
  cadenas terminadas: 50/100
  cadenas terminadas: 60/100
  cadenas terminadas: 70/100
  cadenas terminadas: 80/100
  cadenas terminadas: 90/100


KeyboardInterrupt: 

In [4]:
"""
PASO 3b - Busqueda de picos en M(d)
===================================
Tesis: Complejidad e informacion en un minority game con opcion de abstencion

NO vuelve a simular. Lee los CSV que genero el Paso 3
(Paso3_Md_<etiqueta>_N<N>_promedio.csv), asi que corre en segundos.
Debe estar en la MISMA carpeta que esos CSV.

Que hace, para cada grupo de N:
  1. Grafica completa (d = 1..100) con los picos significativos etiquetados.
  2. Usa un corte fijo en d = 10 (D_CORTE) para descartar el pico grande.
     Si se pone D_CORTE = None, el corte se calcula solo: el primer d en
     que TODAS las curvas del grupo regresan a la linea de ruido.
  3. Dos figuras de 4 paneles (un panel por N), con el eje vertical
     ajustado a cada ventana:
        ventana 1: de d = 10 a d = 50
        ventana 2: de d = 50 a d = 100
     Con lineas guia tenues en d = 27, 54, 81 (P, 2P, 3P; P = 3^m = 27).
  4. Imprime y guarda una tabla con todos los picos encontrados.

Que cuenta como PICO SIGNIFICATIVO:
  - es un maximo local de la curva promedio, y
  - su intervalo de confianza del 95% queda COMPLETO por encima de la
    linea de ruido (el sesgo esperado de una cadena sin correlacion).
  Ojo: con intervalos del 95%, por puro azar se espera que ~2-3 de los
  100 valores de d de cada curva queden por encima. Un pico aislado no
  basta; lo que cuenta es que se repita (en varios N, o en multiplos de 27).
"""

import os
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch


# =====================================================================
# PARAMETROS  (deben coincidir con los del Paso 3)
# =====================================================================
ETIQUETA = "con_puntaje"     # "con_puntaje" o "sin_puntaje"
DESCRIPCION = ("2 estrategias, con puntaje" if ETIQUETA == "con_puntaje"
               else "2 estrategias, sin puntaje")

M = 3                        # memoria
K = 3                        # tamano del alfabeto
P = K ** M                   # numero de historias posibles (27)

GRUPOS = [
    ("grupo1", [100, 200, 300, 400]),
    ("grupo2", [1000, 2000, 3000, 4000]),
]

D_CORTE = 10                 # inicio de la ventana 1 (fijo). None = automatico
LIMITE_VENTANAS = 50         # ventana 1: corte..50, ventana 2: 50..100
GUIAS = [P, 2 * P, 3 * P]    # lineas guia: 27, 54, 81

COLORES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]   # uno por cada N
GRIS = "#52514e"
TINTA = "#0b0b0b"

plt.rcParams.update({
    "font.size": 11,
    "axes.edgecolor": GRIS,
    "axes.labelcolor": TINTA,
    "xtick.color": GRIS,
    "ytick.color": GRIS,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


# =====================================================================
# LECTURA DE LOS RESULTADOS DEL PASO 3
# =====================================================================
def leer_N(N):
    """Lee el CSV de promedios de un N. Devuelve None si no existe."""
    archivo = f"Paso3_Md_{ETIQUETA}_N{N}_promedio.csv"
    if not os.path.exists(archivo):
        print(f"  FALTA el archivo {archivo}. Corre primero el Paso 3.")
        return None
    # columnas: d, media, desv, error_est, ic_inf, ic_sup, sesgo
    tabla = np.loadtxt(archivo, delimiter=",", comments="#")
    return {
        "d":      tabla[:, 0].astype(int),
        "media":  tabla[:, 1],
        "ic_inf": tabla[:, 4],
        "ic_sup": tabla[:, 5],
        "sesgo":  tabla[:, 6],
    }


# =====================================================================
# PICOS Y CORTE
# =====================================================================
def encontrar_picos(r):
    """Indices de los picos significativos de una curva.

    Pico = maximo local de la media cuyo intervalo de confianza queda
    completo por encima de la linea de ruido.
    """
    media = r["media"]
    n = len(media)
    indices = []
    for i in range(n):
        izquierda = media[i - 1] if i > 0 else -np.inf
        derecha = media[i + 1] if i < n - 1 else -np.inf
        es_maximo = media[i] >= izquierda and media[i] > derecha
        es_significativo = r["ic_inf"][i] > r["sesgo"][i]
        if es_maximo and es_significativo:
            indices.append(i)
    return np.array(indices, dtype=int)


def fin_del_pico_grande(r):
    """Primer d, despues del maximo global, en que el intervalo de
    confianza toca la linea de ruido (la curva 'regresa' al ruido)."""
    i_max = int(np.argmax(r["media"]))
    for i in range(i_max + 1, len(r["d"])):
        if r["ic_inf"][i] <= r["sesgo"][i]:
            return int(r["d"][i])
    return int(r["d"][-1])


# =====================================================================
# AYUDAS DE GRAFICADO
# =====================================================================
def etiquetar_picos(ax, r, indices, color):
    """Marca cada pico con un punto y escribe su d encima."""
    for k, i in enumerate(indices):
        ax.plot(r["d"][i], r["media"][i], marker="o", ms=6, color=color,
                markeredgecolor="white", markeredgewidth=1, zorder=5)
        ax.annotate(f"{r['d'][i]}", (r["d"][i], r["media"][i]),
                    textcoords="offset points",
                    xytext=(0, 6 + 7 * (k % 2)),   # alterna altura
                    ha="center", fontsize=7.5, color=color,
                    annotation_clip=False)


def leyenda_ventanas():
    manijas = [
        Line2D([], [], color=GRIS, lw=1.5, marker="o", ms=3),
        Patch(color=GRIS, alpha=0.25, lw=0),
        Line2D([], [], color=GRIS, lw=1.2, ls="--"),
        Line2D([], [], color=GRIS, lw=0.8, ls=":"),
        Line2D([], [], color=GRIS, lw=0, marker="o", ms=6,
               markeredgecolor="white"),
    ]
    etiquetas = [
        "Promedio de 100 cadenas",
        "Intervalo de confianza 95%",
        "Sesgo esperado de una cadena sin correlación",
        f"Guías en d = {', '.join(str(g) for g in GUIAS)}  (múltiplos de P = {P})",
        "Pico significativo (número = d)",
    ]
    return manijas, etiquetas


# =====================================================================
# GRAFICA 1: COMPLETA, CON PICOS ETIQUETADOS
# =====================================================================
def graficar_completa(nombre, lista_N, datos, picos, corte):
    fig, ax = plt.subplots(figsize=(10, 5.5))
    for N, color in zip(lista_N, COLORES):
        if N not in datos:
            continue
        r = datos[N]
        ax.plot(r["d"], r["media"], color=color, lw=1.8, marker="o", ms=3,
                label=f"N = {N}  (α = {P / N:.4f})")
        etiquetar_picos(ax, r, picos[N], color)
    r0 = datos[next(iter(datos))]
    ax.plot(r0["d"], r0["sesgo"], color=GRIS, lw=1.5, ls="--",
            label="Sesgo esperado de una cadena sin correlación")
    ax.axvline(corte, color=TINTA, lw=1, ls="-.", alpha=0.7,
               label=f"Corte común: d = {corte}")
    ax.set_xlabel("Distancia d")
    ax.set_ylabel("M(d)  [bits]")
    ax.set_title(f"M(d) con picos significativos — {DESCRIPCION}\n"
                 f"N = {', '.join(str(n) for n in lista_N)};  "
                 f"número sobre cada punto = d del pico",
                 loc="left", color=TINTA)
    ax.set_xlim(0, r0["d"][-1] + 1)
    ax.set_ylim(bottom=0)
    ax.margins(y=0.12)
    ax.grid(axis="y", color="#e5e4df", lw=0.8)
    ax.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
    ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.13),
              ncol=3, fontsize=9)
    fig.tight_layout()
    archivo = f"Paso3b_{ETIQUETA}_{nombre}_completa_con_picos.png"
    fig.savefig(archivo, dpi=200)
    plt.close(fig)
    print(f"  Guardado: {archivo}")


# =====================================================================
# GRAFICAS 2 Y 3: VENTANAS, UN PANEL POR N
# =====================================================================
def graficar_ventana(nombre, lista_N, datos, picos, a, b):
    fig, ejes = plt.subplots(2, 2, figsize=(12, 7.5), sharex=True)
    for ax, N, color in zip(ejes.flat, lista_N, COLORES):
        if N not in datos:
            ax.set_visible(False)
            continue
        r = datos[N]
        mascara = (r["d"] >= a) & (r["d"] <= b)
        d = r["d"][mascara]

        ax.fill_between(d, r["ic_inf"][mascara], r["ic_sup"][mascara],
                        color=color, alpha=0.18, lw=0)
        ax.plot(d, r["media"][mascara], color=color, lw=1.5, marker="o", ms=2.5)
        ax.plot(d, r["sesgo"][mascara], color=GRIS, lw=1.2, ls="--")
        for g in GUIAS:
            if a <= g <= b:
                ax.axvline(g, color=GRIS, lw=0.8, ls=":", alpha=0.7)

        en_ventana = [i for i in picos[N] if a <= r["d"][i] <= b]
        etiquetar_picos(ax, r, en_ventana, color)

        # eje vertical ajustado SOLO a lo que se ve en esta ventana
        y_min = min(r["ic_inf"][mascara].min(), r["sesgo"][mascara].min())
        y_max = max(r["ic_sup"][mascara].max(), r["sesgo"][mascara].max())
        rango = y_max - y_min if y_max > y_min else abs(y_max) * 0.1 + 1e-9
        ax.set_ylim(y_min - 0.05 * rango, y_max + 0.20 * rango)
        ax.set_xlim(a - 0.5, b + 0.5)

        ax.set_title(f"N = {N}   (α = {P / N:.4f})", loc="left",
                     fontsize=10, color=TINTA)
        ax.grid(axis="y", color="#e5e4df", lw=0.8)
        ax.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))

    for ax in ejes.flat[len(lista_N):]:
        ax.set_visible(False)
    for ax in ejes[1]:
        ax.set_xlabel("Distancia d")
    for ax in ejes[:, 0]:
        ax.set_ylabel("M(d)  [bits]")

    fig.suptitle(f"M(d) entre d = {a} y d = {b} — {DESCRIPCION}\n"
                 f"(cada panel tiene su propia escala vertical)",
                 x=0.01, ha="left", fontsize=12, color=TINTA)
    manijas, etiquetas = leyenda_ventanas()
    fig.legend(manijas, etiquetas, loc="lower center", ncol=3,
               frameon=False, fontsize=9)
    fig.tight_layout(rect=[0, 0.08, 1, 0.93])
    archivo = f"Paso3b_{ETIQUETA}_{nombre}_ventana_d{a}_a_d{b}.png"
    fig.savefig(archivo, dpi=200)
    plt.close(fig)
    print(f"  Guardado: {archivo}")


# =====================================================================
# PROGRAMA PRINCIPAL
# =====================================================================
if __name__ == "__main__":

    filas_tabla = []

    for nombre, lista_N in GRUPOS:
        print("\n" + "=" * 62)
        print(f"  {nombre}: N = {lista_N}   ({DESCRIPCION})")
        print("=" * 62)

        datos = {}
        for N in lista_N:
            r = leer_N(N)
            if r is not None:
                datos[N] = r
        if not datos:
            print("  No hay datos para este grupo; se omite.")
            continue

        # --- picos y fin del pico grande de cada N ---
        picos, fines = {}, {}
        for N, r in datos.items():
            picos[N] = encontrar_picos(r)
            fines[N] = fin_del_pico_grande(r)

        # --- corte comun del grupo ---
        if D_CORTE is None:
            corte = max(fines.values())
            print(f"\n  Fin del pico grande por N: "
                  + ", ".join(f"N={N}: d={fines[N]}" for N in fines))
            print(f"  Corte comun (automatico): d = {corte}")
        else:
            corte = int(D_CORTE)
            print(f"\n  Corte comun (fijado a mano): d = {corte}")

        # --- tabla de picos ---
        print("\n  Picos significativos (d : M(d) en bits):")
        for N, r in datos.items():
            texto = ", ".join(f"{r['d'][i]}:{r['media'][i]:.2e}"
                              for i in picos[N])
            print(f"    N={N:5d}  ->  {texto if texto else '(ninguno)'}")
            for i in picos[N]:
                filas_tabla.append(
                    f"{nombre},{N},{P / N:.6f},{r['d'][i]},"
                    f"{r['media'][i]:.8e},{r['ic_inf'][i]:.8e},"
                    f"{r['ic_sup'][i]:.8e},{r['sesgo'][i]:.8e},"
                    f"{r['media'][i] - r['sesgo'][i]:.8e},"
                    f"{'si' if r['d'][i] < corte else 'no'}"
                )

        # --- graficas ---
        print()
        graficar_completa(nombre, lista_N, datos, picos, corte)
        d_ultimo = int(datos[next(iter(datos))]["d"][-1])
        if corte < LIMITE_VENTANAS:
            graficar_ventana(nombre, lista_N, datos, picos,
                             corte, LIMITE_VENTANAS)
            graficar_ventana(nombre, lista_N, datos, picos,
                             LIMITE_VENTANAS, d_ultimo)
        else:
            print(f"  AVISO: el corte (d={corte}) es mayor o igual que "
                  f"{LIMITE_VENTANAS}; solo se hace la ventana "
                  f"d={corte} a d={d_ultimo}.")
            graficar_ventana(nombre, lista_N, datos, picos, corte, d_ultimo)

    # --- tabla de todos los picos en un CSV ---
    archivo = f"Paso3b_picos_{ETIQUETA}.csv"
    with open(archivo, "w", encoding="utf-8") as f:
        f.write(f"# PASO 3b. Picos significativos de M(d), {DESCRIPCION}\n")
        f.write("# pico = maximo local con IC 95% completo sobre el sesgo\n")
        f.write("grupo,N,alfa,d,media,ic_inferior,ic_superior,sesgo,"
                "exceso_sobre_sesgo,dentro_del_pico_grande\n")
        for fila in filas_tabla:
            f.write(fila + "\n")
    print(f"\n  Guardado: {archivo}")
    print("\n  Listo.")


  grupo1: N = [100, 200, 300, 400]   (2 estrategias, con puntaje)

  Corte comun (fijado a mano): d = 10

  Picos significativos (d : M(d) en bits):
    N=  100  ->  2:1.13e-02, 11:5.96e-04, 17:5.92e-04, 24:5.71e-04, 34:6.08e-04
    N=  200  ->  4:3.73e-03, 6:1.04e-03, 9:6.49e-04, 11:6.20e-04, 13:5.82e-04, 16:5.91e-04, 20:6.21e-04, 22:5.85e-04, 26:5.91e-04, 44:5.58e-04, 63:5.60e-04, 92:5.64e-04
    N=  300  ->  4:4.58e-03, 6:1.00e-03, 9:6.01e-04, 12:7.38e-04, 17:6.79e-04, 20:6.88e-04, 22:6.02e-04, 24:6.44e-04, 27:5.79e-04, 30:6.50e-04, 33:6.05e-04, 35:5.75e-04, 42:5.75e-04, 44:5.83e-04, 47:5.94e-04, 49:5.65e-04, 56:5.76e-04, 96:5.73e-04
    N=  400  ->  4:5.44e-03, 6:1.06e-03, 9:6.90e-04, 13:7.59e-04, 15:6.86e-04, 20:6.48e-04, 23:6.73e-04, 25:6.85e-04, 28:6.34e-04, 30:6.30e-04, 34:6.47e-04, 37:6.14e-04, 39:6.05e-04, 95:5.85e-04

  Guardado: Paso3b_con_puntaje_grupo1_completa_con_picos.png
  Guardado: Paso3b_con_puntaje_grupo1_ventana_d10_a_d50.png
  Guardado: Paso3b_con_puntaje_grupo1